In [1]:
from tritonclient.http import InferenceServerClient
from tritonhttpclient import InferInput
import numpy as np
import json

/home/yrlab/miniconda3/envs/hf/lib/python3.12/site-packages/tritonhttpclient/__init__.py:33: DeprecationWarning: The package `tritonhttpclient` is deprecated and will be removed in a future version. Please use instead `tritonclient.http`
  warnings.warn(


config input / output
```
input [
  {
    name: "text"
    data_type: TYPE_STRING
    dims: [ -1 ] 
  }
]
output [
  {
    name: "spans"
    data_type: TYPE_STRING
    dims: [ -1 ]
  },
  {
    name: "predictions"
    data_type: TYPE_STRING
    dims: [ -1 ]
  }
]

```

In [2]:
# https://huggingface.co/datasets/stanfordnlp/imdb/viewer/plain_text/train?row=2
texts = [
    "안녕하세요, 저는 삼성전자에 지원한 김삼정입니다. 박선호 교수님 연구실에서 반도체 소재 연구를 했습니다.",
    "신청자 홍길동(RRN 900101-1234567)님의 계좌번호는 123-456-789012 입니다.",
    # "문의사항은 kim.chulsoo@example.com 또는 010-1234-5678 로 연락 주세요.",
    # "배송지: 서울특별시 강남구 테헤란로 123, 우편번호 06134",
    # "카드번호 4111-1111-1111-1111, 유효기간 12/27, CVC 123 결제 완료.",
    # "아이디 pii_tester_01 로 로그인 후 비밀번호는 P@ssw0rd!2024 로 변경했습니다.",
    # "여권번호 M12345678, 운전면허번호 11-22-334455-66 확인 부탁드립니다.",
]

request_input = InferInput(
    "text",
    [len(texts), 1],
    "BYTES"
)

input_data = np.array(texts, dtype=object)[:, np.newaxis]
request_input.set_data_from_numpy(input_data)
inputs = [request_input]

In [3]:
url = "localhost:8000"
model_name = "span-detection-vllm-example"

with InferenceServerClient(url=url) as client:
    results = client.infer(model_name=model_name, inputs=inputs)

In [4]:
spans = results.as_numpy("spans").tolist()
spans = [json.loads(x) for x in spans]
print(json.dumps(spans, indent=2, ensure_ascii=False))

[
  [
    {
      "tag": "PERSON",
      "begin": 20,
      "end": 23,
      "value": "김삼정"
    },
    {
      "tag": "PERSON",
      "begin": 28,
      "end": 31,
      "value": "박선호"
    }
  ],
  [
    {
      "tag": "PERSON",
      "begin": 4,
      "end": 7,
      "value": "홍길동"
    },
    {
      "tag": "RRN",
      "begin": 12,
      "end": 26,
      "value": "900101-1234567"
    },
    {
      "tag": "ACCOUNT_NUMBER",
      "begin": 36,
      "end": 50,
      "value": "123-456-789012"
    }
  ]
]


In [9]:
predictions = results.as_numpy("predictions").tolist()

sample_predictions = json.loads(predictions[0])

for prediction in sample_predictions[:8]:
    print(prediction)

{'token_id': 31805, 'value': '안녕', 'label': 'O', 'begin': 0, 'end': 2}
{'token_id': 20327, 'value': '##하', 'label': 'O', 'begin': 2, 'end': 3}
{'token_id': 32541, 'value': '##세요', 'label': 'O', 'begin': 3, 'end': 5}
{'token_id': 48, 'value': ',', 'label': 'O', 'begin': 5, 'end': 6}
{'token_id': 16819, 'value': '저', 'label': 'O', 'begin': 7, 'end': 8}
{'token_id': 20430, 'value': '##는', 'label': 'O', 'begin': 8, 'end': 9}
{'token_id': 35817, 'value': '삼성전자', 'label': 'O', 'begin': 10, 'end': 14}
{'token_id': 20406, 'value': '##에', 'label': 'O', 'begin': 14, 'end': 15}
